In [ ]:
# baseline model TF-IDF feature extraction followed by LinearSVC classifier
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# load dataset
df = pd.read_csv("Labelled_dataset.csv", encoding="utf-8-sig")

# simple text cleaning
def simple_clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# remove missing values
df = df.dropna(subset=["comment_text_clean", "label"]).copy()

# preprocess text
df["text_processed"] = df["comment_text_clean"].apply(simple_clean)

# standardize labels
df["label"] = df["label"].astype(str).str.strip().str.lower()

# updated valid labels for 3-class classification
valid_labels = ["pro", "anti", "neutral"]

# keep only valid labels
df = df[df["label"].isin(valid_labels)].copy()

# features and target
X = df["text_processed"]
y = df["label"]

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# model pipeline
model = Pipeline([
    ("features", FeatureUnion([
        ("word_features", TfidfVectorizer(
            ngram_range=(1, 2),
            sublinear_tf=True,
            max_features=20000
        )),
        ("char_features", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            max_features=20000
        ))
    ])),
    ("clf", LinearSVC(
        class_weight="balanced",
        C=1.0,
        random_state=42,
        max_iter=10000
    ))
])

# train model
model.fit(X_train, y_train)

# predictions
y_pred = model.predict(X_test)

# evaluation
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
#CNN
import re
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from collections import Counter
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CSV_PATH = 'Labelled_youtube_comments.csv'
TEXT_COL = 'comment_text_clean'
LABEL_COL = 'label'

# updated 3 classes
VALID_LABELS = ['anti', 'neutral', 'pro']

MAX_VOCAB = 20000
MAX_LEN = 60
BATCH_SIZE = 32
EMB_DIM = 200
NUM_FILTERS = 128
EPOCHS = 15
LR = 1e-3
DROPOUT = 0.5
PATIENCE = 4

# multilingual-friendly cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

# load dataset
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()

df[TEXT_COL] = df[TEXT_COL].astype(str)

df[LABEL_COL] = (
    df[LABEL_COL]
    .astype(str)
    .str.strip()
    .str.lower()
)

# keep only valid labels
df = df[df[LABEL_COL].isin(VALID_LABELS)].copy()

# label encoding
label_to_idx = {lab: i for i, lab in enumerate(VALID_LABELS)}
idx_to_label = {i: lab for lab, i in label_to_idx.items()}

df['label_id'] = df[LABEL_COL].map(label_to_idx)

# tokenize
df['tokens'] = df[TEXT_COL].apply(tokenize)

# train / validation / test split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df['label_id']
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df['label_id']
)

# build vocabulary
counter = Counter()

for toks in train_df['tokens']:
    counter.update(toks)

itos = ['<pad>', '<unk>']

for word, freq in counter.most_common(MAX_VOCAB - len(itos)):
    itos.append(word)

stoi = {w: i for i, w in enumerate(itos)}

# encoding function
def encode(tokens):
    ids = [stoi.get(tok, stoi['<unk>']) for tok in tokens][:MAX_LEN]

    if len(ids) < MAX_LEN:
        ids += [stoi['<pad>']] * (MAX_LEN - len(ids))

    return ids

# dataset class
class CommentDataset(Dataset):

    def __init__(self, frame):
        self.texts = frame['tokens'].tolist()
        self.labels = frame['label_id'].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

# datasets
train_ds = CommentDataset(train_df)
val_ds = CommentDataset(val_df)
test_ds = CommentDataset(test_df)

# dataloaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# CNN model
class TextCNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        emb_dim,
        num_classes,
        num_filters=128,
        kernel_sizes=(3,4,5),
        dropout=0.5
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            emb_dim,
            padding_idx=0
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(emb_dim, num_filters, k)
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

    def forward(self, x):

        x = self.embedding(x).transpose(1, 2)

        feats = []

        for conv in self.convs:
            h = torch.relu(conv(x))
            p = torch.max(h, dim=2).values
            feats.append(p)

        x = torch.cat(feats, dim=1)

        x = self.dropout(x)

        return self.fc(x)

# class weights
counts = train_df['label_id'].value_counts().sort_index().values

weights = counts.sum() / (len(counts) * counts)

class_weights = torch.tensor(
    weights,
    dtype=torch.float32
).to(DEVICE)

# model
model = TextCNN(
    len(itos),
    EMB_DIM,
    len(VALID_LABELS),
    NUM_FILTERS,
    dropout=DROPOUT
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=1
)

# evaluation function
def eval_model(loader):

    model.eval()

    ys = []
    preds = []

    total_loss = 0.0

    with torch.no_grad():

        for x, y in loader:

            x, y = x.to(DEVICE), y.to(DEVICE)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)

            pred = logits.argmax(dim=1)

            ys.extend(y.cpu().numpy().tolist())
            preds.extend(pred.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)

    acc = accuracy_score(ys, preds)

    f1 = f1_score(
        ys,
        preds,
        average='macro',
        zero_division=0
    )

    return avg_loss, acc, f1, ys, preds

# training loop
best_f1 = -1
best_state = None
pat = 0

history = []

for epoch in range(1, EPOCHS + 1):

    model.train()

    train_loss = 0.0

    for x, y in train_loader:

        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        optimizer.step()

        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)

    val_loss, val_acc, val_f1, _, _ = eval_model(val_loader)

    scheduler.step(val_f1)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'val_f1': val_f1
    })

    print(
        f'Epoch {epoch:02d} | '
        f'train_loss={train_loss:.4f} | '
        f'val_loss={val_loss:.4f} | '
        f'val_acc={val_acc:.4f} | '
        f'val_f1={val_f1:.4f}'
    )

    if val_f1 > best_f1:

        best_f1 = val_f1

        best_state = {
            k: v.cpu().clone()
            for k, v in model.state_dict().items()
        }

        pat = 0

    else:

        pat += 1

        if pat >= PATIENCE:
            print('Early stopping triggered.')
            break

# load best model
if best_state is not None:
    model.load_state_dict(best_state)

# final evaluation
val_loss, val_acc, val_f1, val_y, val_pred = eval_model(val_loader)

test_loss, test_acc, test_f1, test_y, test_pred = eval_model(test_loader)

print('\nValidation:')
print(f'loss={val_loss:.4f} acc={val_acc:.4f} macro_f1={val_f1:.4f}')

print('\nTest:')
print(f'loss={test_loss:.4f} acc={test_acc:.4f} macro_f1={test_f1:.4f}')

print('\nClassification Report:')
print(classification_report(
    test_y,
    test_pred,
    target_names=VALID_LABELS,
    digits=4,
    zero_division=0
))

print('\nConfusion Matrix:')
print(confusion_matrix(test_y, test_pred))

# save outputs
out = 'output'

os.makedirs(out, exist_ok=True)

pd.DataFrame(history).to_csv(
    f'{out}/cnn_training_history.csv',
    index=False
)

with open(f'{out}/cnn_test_metrics.json', 'w') as f:

    json.dump({
        'test_acc': float(test_acc),
        'test_macro_f1': float(test_f1),
        'best_val_macro_f1': float(best_f1)
    }, f, indent=2)

In [ ]:
#mBERT
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# multilingual BERT
MODEL_NAME = "bert-base-multilingual-cased"

CSV_PATH = "Labelled_youtube_comments.csv"

TEXT_COL = "comment_text_clean"
LABEL_COL = "label"

# updated 3-class setup
VALID_LABELS = ["anti", "neutral", "pro"]

MAX_LENGTH = 128

TEST_SIZE = 0.2
VAL_SIZE = 0.2

OUTPUT_DIR = "output/mbert_model"

# label mappings
label2id = {
    lab: i
    for i, lab in enumerate(VALID_LABELS)
}

id2label = {
    i: lab
    for lab, i in label2id.items()
}

# minimal cleaning
def clean_text(text):

    text = str(text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

# load dataset
df = pd.read_csv(
    CSV_PATH,
    encoding="utf-8-sig"
)

# remove missing values
df = df.dropna(
    subset=[TEXT_COL, LABEL_COL]
).copy()

# clean text
df[TEXT_COL] = (
    df[TEXT_COL]
    .astype(str)
    .apply(clean_text)
)

# normalize labels
df[LABEL_COL] = (
    df[LABEL_COL]
    .astype(str)
    .str.strip()
    .str.lower()
)

# keep only valid labels
df = df[
    df[LABEL_COL].isin(VALID_LABELS)
].copy()

# label encoding
df["label_id"] = df[LABEL_COL].map(label2id)

# train-test split
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df["label_id"]
)

# train-validation split
train_df, val_df = train_test_split(
    train_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_df["label_id"]
)

print("Train/Val/Test sizes:")
print(len(train_df), len(val_df), len(test_df))

print("\nTrain label distribution:")
print(train_df[LABEL_COL].value_counts())

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# dataset class
class TextDataset(Dataset):

    def __init__(self, frame):

        self.texts = frame[TEXT_COL].tolist()

        self.labels = frame["label_id"].tolist()

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False
        )

        enc["labels"] = self.labels[idx]

        return enc

# datasets
train_ds = TextDataset(train_df)

val_ds = TextDataset(val_df)

test_ds = TextDataset(test_df)

# dynamic padding
collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

# model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(VALID_LABELS),
    id2label=id2label,
    label2id=label2id
)

# batch sizes
use_cuda = torch.cuda.is_available()

per_device_train_batch_size = 8 if use_cuda else 4

per_device_eval_batch_size = 16 if use_cuda else 4

# training on CPU, so epochs manually set to 5
# if using GPU later, you can use:
# num_train_epochs = 3 if use_cuda else 3
num_train_epochs = 3

# training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    learning_rate=2e-5,

    per_device_train_batch_size=per_device_train_batch_size,

    per_device_eval_batch_size=per_device_eval_batch_size,

    num_train_epochs=num_train_epochs,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",

    greater_is_better=True,

    logging_strategy="epoch",

    report_to="none",

    seed=SEED
)

# class weights
from collections import Counter

counts = Counter(
    train_df["label_id"].tolist()
)

class_weights = torch.tensor(
    [
        len(train_df) /
        (len(counts) * counts[i])

        for i in range(len(VALID_LABELS))
    ],
    dtype=torch.float
)

# custom trainer with weighted loss
class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(model.device)
        )

        loss = loss_fct(logits, labels)

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

# evaluation metrics
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),

        "f1_macro": f1_score(
            labels,
            preds,
            average="macro",
            zero_division=0
        )
    }

# trainer
trainer = WeightedTrainer(
    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=val_ds,

    processing_class=tokenizer,

    data_collator=collator,

    compute_metrics=compute_metrics
)

# train model
trainer.train()

# validation evaluation
val_metrics = trainer.evaluate(val_ds)

# test predictions
test_pred = trainer.predict(test_ds)

test_logits = test_pred.predictions

test_labels = test_pred.label_ids

test_preds = np.argmax(test_logits, axis=1)

# print results
print("\nValidation metrics:")
print(val_metrics)

print("\nTest metrics:")
print({
    "accuracy": accuracy_score(
        test_labels,
        test_preds
    ),

    "f1_macro": f1_score(
        test_labels,
        test_preds,
        average="macro",
        zero_division=0
    )
})

print("\nClassification Report:")

print(classification_report(
    test_labels,
    test_preds,
    target_names=VALID_LABELS,
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix:")

print(confusion_matrix(
    test_labels,
    test_preds
))

# save outputs
os.makedirs("output", exist_ok=True)

trainer.save_model(OUTPUT_DIR)

with open(
    "output/mbert_test_metrics.json",
    "w"
) as f:

    json.dump({

        "val_metrics": {
            k: float(v)

            for k, v in val_metrics.items()

            if isinstance(
                v,
                (int, float, np.floating)
            )
        },

        "test_accuracy": float(
            accuracy_score(
                test_labels,
                test_preds
            )
        ),

        "test_macro_f1": float(
            f1_score(
                test_labels,
                test_preds,
                average="macro",
                zero_division=0
            )
        )

    }, f, indent=2)

In [1]:
# temporal analysis

import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px

INPUT_CSV = 'Labelled_dataset.csv'

TEXT_COL = 'comment_text_clean'
LABEL_COL = 'label'
DATE_COL = 'published_utc'

# updated 3-class labels
VALID_LABELS = ['anti', 'neutral', 'pro']

out = Path('output')
out.mkdir(exist_ok=True)

# simple cleaning
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)

    text = re.sub(r'@\w+', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

# load dataset
df = pd.read_csv(
    INPUT_CSV,
    encoding='utf-8-sig'
)

# remove missing values
df = df.dropna(
    subset=[TEXT_COL, LABEL_COL, DATE_COL]
).copy()

# clean text
df[TEXT_COL] = (
    df[TEXT_COL]
    .astype(str)
    .apply(clean_text)
)

# normalize labels
df[LABEL_COL] = (
    df[LABEL_COL]
    .astype(str)
    .str.strip()
    .str.lower()
)

# keep only valid labels
df = df[
    df[LABEL_COL].isin(VALID_LABELS)
].copy()

# parse timestamps
try:

    df[DATE_COL] = pd.to_datetime(
        df[DATE_COL],
        utc=True,
        errors='coerce'
    )

except Exception:

    df[DATE_COL] = pd.to_datetime(
        df[DATE_COL],
        errors='coerce'
    )

# keep valid timestamps
valid_ts = df[DATE_COL].notna().sum()

print(f'Valid timestamps: {valid_ts} / {len(df)}')

df = df[df[DATE_COL].notna()].copy()

# remove timezone
df['published_dt'] = (
    df[DATE_COL]
    .dt.tz_convert(None)
)

# quarterly aggregation works better for small datasets
df['year_quarter'] = (
    df['published_dt']
    .dt.to_period('Q')
    .astype(str)
)

df['year'] = (
    df['published_dt']
    .dt.year
)

# helper for saving figures
def save_plotly(fig, path):

    try:

        fig.write_image(
            str(path),
            width=1400,
            height=800,
            scale=2
        )

    except Exception as e:

        print(
            f'Could not save image {path.name}: {e}'
        )

# 1) overall label distribution
label_counts = (
    df[LABEL_COL]
    .value_counts()
    .reset_index()
)

label_counts.columns = ['label', 'count']

label_counts.to_csv(
    out / 'label_distribution.csv',
    index=False
)

fig = px.bar(
    label_counts,
    x='label',
    y='count',
    title='Overall label distribution'
)

fig.update_layout(
    xaxis_title='Label',
    yaxis_title='Count'
)

save_plotly(
    fig,
    out / 'label_distribution.png'
)

# 2) quarterly label counts
quarterly = (
    df.groupby(['year_quarter', LABEL_COL])
    .size()
    .reset_index(name='count')
)

quarterly.to_csv(
    out / 'quarterly_label_counts.csv',
    index=False
)

fig = px.line(
    quarterly,
    x='year_quarter',
    y='count',
    color=LABEL_COL,
    markers=True,
    title='Quarterly label counts over time'
)

fig.update_layout(
    xaxis_title='Quarter',
    yaxis_title='Count'
)

save_plotly(
    fig,
    out / 'quarterly_label_counts.png'
)

# 3) peak quarter per label
peak_quarters = (
    quarterly.loc[
        quarterly.groupby(LABEL_COL)['count'].idxmax()
    ]
    .sort_values(LABEL_COL)
)

peak_quarters.to_csv(
    out / 'peak_quarter_per_label.csv',
    index=False
)

# 4) summary
summary = {

    'rows_after_date_filter': int(len(df)),

    'valid_timestamp_rows': int(valid_ts),

    'unique_quarters': int(
        df['year_quarter'].nunique()
    ),

    'label_counts': (
        df[LABEL_COL]
        .value_counts()
        .to_dict()
    ),

    'date_start': str(
        df['published_dt'].min()
    ),

    'date_end': str(
        df['published_dt'].max()
    )
}

pd.Series(summary).to_json(
    out / 'temporal_summary.json',
    indent=2
)

print("\nSummary:\n")

print(summary)

print("\nPeak quarter per label:\n")

print(peak_quarters)

Valid timestamps: 320 / 320

Summary:

{'rows_after_date_filter': 320, 'valid_timestamp_rows': 320, 'unique_quarters': 14, 'label_counts': {'neutral': 110, 'anti': 105, 'pro': 105}, 'date_start': '2020-07-07 09:45:32', 'date_end': '2026-03-30 03:24:12'}

Peak quarter per label:

   year_quarter    label  count
19       2024Q2     anti     44
6        2021Q2  neutral     56
7        2021Q2      pro     57


In [8]:
# temporal visualization

import pandas as pd
import plotly.express as px
from pathlib import Path

out = Path('output')

out.mkdir(exist_ok=True)

# load temporal outputs
quarterly = pd.read_csv(
    out / 'quarterly_label_counts.csv'
)

peak = pd.read_csv(
    out / 'peak_quarter_per_label.csv'
)

# convert quarter to datetime
quarterly['year_quarter_dt'] = pd.PeriodIndex(
    quarterly['year_quarter'],
    freq='Q'
).to_timestamp()

quarterly = quarterly.sort_values(
    'year_quarter_dt'
)

# 1) stacked bar chart
fig2 = px.bar(
    quarterly,
    x='year_quarter_dt',
    y='count',
    color='label',
    title='Quarterly label counts (stacked bar)'
)

fig2.update_layout(
    xaxis_title='Quarter',
    yaxis_title='Count',
    barmode='stack',
    legend_title='Label'
)

fig2.show()

# 2) heatmap
heat = quarterly.pivot(
    index='label',
    columns='year_quarter_dt',
    values='count'
).fillna(0)

fig3 = px.imshow(
    heat,
    aspect='auto',
    color_continuous_scale='Blues',
    title='Temporal heatmap of quarterly label counts'
)

fig3.update_layout(
    xaxis_title='Quarter',
    yaxis_title='Label'
)

fig3.show()

# peak periods
print("\nPeak quarter per label:\n")

print(peak)


Peak quarter per label:

  year_quarter    label  count
0       2024Q2     anti     44
1       2021Q2  neutral     56
2       2021Q2      pro     57
